In [0]:
-- ===========================================================================================================================
-- MULTIMODAL TRAFFIC - GOLD LAYER
-- SQL ANALYTICS / BUSINESS QUESTIONS
-- ===========================================================================================================================
-- Purpose :
-- Use the curated Gold dimensional and fact tables to perform analytical SQL and answer business-oriented questions
-- about network bandwidth utilization, site performance, latency, and regional traffic distribution.
-- Dataset grain :
--   - dim_date: One record represents one unique calendar date with temporal attributes.
--   - dim_time: One record represents a specific time of day with hour, minute, and period granularity.
--   - dim_site: One record represents a physical network site/location with regional and operational metadata.
--   - dim_transport: One record represents a transport protocol or connection type with provider and capacity specifications.
--   - fact_traffic: One record represents network traffic metrics (bytes, latency, packet loss) for a given site, transport type, date, and time.
-- Main columns :
--   - dim_date: date_key, full_date, day_of_week, is_weekend, month_name, year
--   - dim_time: time_key, hour_of_day, minute_of_hour, time_period
--   - dim_site: site_key, site_name, region, status
--   - dim_transport: transport_key, transport_type, provider_name, capacity_mbps
--   - fact_traffic: date_key, time_key, site_key, transport_key, bytes_in, bytes_out, latency_ms, packet_loss_pct
-- Assumed Gold table names :
-- gold.dim_date, gold.dim_time, gold.dim_site, gold.dim_transport, gold.fact_traffic
-- Analytics objective :
-- Move from data preparation to business analysis: monitor bandwidth consumption (bytes in/out), evaluate network latency,
-- assess packet loss percentages, analyze site performance across regions, and optimize capacity planning.
-- ===========================================================================================================================

-- Display Datasets 

SELECT * 
FROM workspace.gold.gold_multimodal_dim_site;

SELECT * 
FROM workspace.gold.gold_multimodal_dim_transport;

SELECT * 
FROM workspace.gold.gold_multimodal_dim_time;

SELECT * 
FROM workspace.gold.gold_multimodal_dim_date
LIMIT 1000;

SELECT * 
FROM workspace.gold.gold_multimodal_fact_traffic
LIMIT 1000;

--  total traffic records  present in the dataset

SELECT 
    COUNT (*) AS total 
FROM workspace.gold.gold_multimodal_fact_traffic;

-- distinct sites in the dataset

SELECT 
    site_label,
    address 
FROM workspace.gold.gold_multimodal_dim_site;

-- the date range available in the date dimension

SELECT 
    MIN (d.full_date) AS min_date,
    MAX (d.full_date) AS max_date
FROM workspace.gold.gold_multimodal_fact_traffic t 
LEFT JOIN workspace.gold.gold_multimodal_dim_date d on t.date_key=d.date_key
;

--  Distinct lane types and directions

SELECT 
    DISTINCT (lane_type) ,
    direction
FROM workspace.gold.gold_multimodal_fact_traffic
;

-- Hours and periods in dim_time 

SELECT 
    time_period,
    COUNT (*) AS total_time_period
FROM workspace.gold.gold_multimodal_dim_time
GROUP BY time_period;

-- transport_key with its transport_mode 

SELECT * 
FROM workspace.gold.gold_multimodal_dim_transport 
;

-- Site locations in dim_transport

SELECT 
    site_id,
    site_label,
    address,
    latitude,
    longitude,
    (site_label = address) AS label_equals_address
FROM workspace.gold.gold_multimodal_dim_site
ORDER BY site_id;

--  Activity per site

SELECT 
    site_id,
    COUNT(site_id) AS site_measure,
    SUM (user_count) AS total_users,
    MIN (date_key) AS first_date,
    MAX (date_key) AS last_date
FROM workspace.gold.gold_multimodal_fact_traffic
GROUP BY site_id;

-- Traffic per site 

SELECT 
    site_label, 
    lane_type,
    SUM (user_count) AS total_traffic
FROM gold.gold_multimodal_fact_traffic t 
LEFT JOIN gold.gold_multimodal_dim_site s on t.site_id=s.site_id
GROUP BY site_label,lane_type
ORDER BY site_label
;

--  total Traffic by transport mode in the dataset 

SELECT 
    transport_mode,
    SUM (user_count) AS total_traffic
FROM workspace.gold.gold_multimodal_fact_traffic f
LEFT JOIN workspace.gold.gold_multimodal_dim_transport t on f.transport_key=t.transport_key
GROUP BY transport_mode;

--  total Traffic by transport mode in one day and in one site 

SELECT 
    transport_mode,
    SUM (user_count) AS total_traffic
FROM workspace.gold.gold_multimodal_fact_traffic f
LEFT JOIN workspace.gold.gold_multimodal_dim_transport t on f.transport_key=t.transport_key
WHERE f.date_key == 20260930 AND site_id == 10015
GROUP BY transport_mode;

-- Avg Weekday versus  avg weekend in one site by transport type in one hour 

SELECT 
    t.transport_mode,
    d.is_weekend,
    ROUND(AVG (user_count),2) AS average_users
FROM workspace.gold.gold_multimodal_fact_traffic f 
LEFT JOIN workspace.gold.gold_multimodal_dim_date d on f.date_key=d.date_key
LEFT JOIN workspace.gold.gold_multimodal_dim_transport t on t.transport_key=f.transport_key
WHERE f.site_id == 10015
GROUP BY  t.transport_mode,d.is_weekend
ORDER BY t.transport_mode;

-- -- Avg Weekday versus  avg weekend in one site by transport type in a day 

WITH  total AS 
(
SELECT 
    t.transport_mode,
    d.is_weekend,
    f.date_key,
    SUM (user_count) AS total_users
FROM workspace.gold.gold_multimodal_fact_traffic f 
LEFT JOIN workspace.gold.gold_multimodal_dim_date d on f.date_key=d.date_key
LEFT JOIN workspace.gold.gold_multimodal_dim_transport t on t.transport_key=f.transport_key
WHERE f.site_id == 10015
GROUP BY f.date_key,t.transport_mode,d.is_weekend
ORDER BY f.date_key DESC ,t.transport_mode DESC
)
SELECT 
    transport_mode,
    is_weekend,
    ROUND(AVG(total_users),2) AS average_users_per_day
FROM total
GROUP BY transport_mode,is_weekend
ORDER BY transport_mode
;

-- Peak windows  in one year 

SELECT 
    time_period,
    SUM (user_count) AS total_traffic
FROM workspace.gold.gold_multimodal_fact_traffic f
LEFT JOIN workspace.gold.gold_multimodal_dim_time t on f.time_key=t.time_key
WHERE f.date_key BETWEEN 20250101 AND 20251231 AND f.site_id == 10015
GROUP BY time_period
;

--  each transport_mode's percentage of total user_coun 

WITH total AS 
(
SELECT 
    t.transport_mode,
    SUM (f.user_count) AS total_users
FROM workspace.gold.gold_multimodal_fact_traffic f 
LEFT JOIN workspace.gold.gold_multimodal_dim_transport t on f.transport_key=t.transport_key
GROUP BY t.transport_mode
)
SELECT 
    transport_mode,
    total_users,
    ROUND(((total_users/SUM(total_users)OVER ())*100),2) AS percentage
FROM total
GROUP BY transport_mode,total_users
;

--  Rank sites by total traffic
